# EuroSAT Binary Water Classification Pipeline

**Task:** Binary classification of Sentinel-2 satellite image patches — Water (1) vs. Other (0)

**Dataset:** [EuroSAT](https://github.com/phelber/EuroSAT) — 27,000 labelled 64×64 patches across 10 land-use classes  
**Target class:** River (index 8) → label 1 | All other classes → label 0  
**Backbone:** ResNet-50 pretrained on ImageNet  
**Framework:** PyTorch Lightning + TorchGeo

---

### Setup Instructions
1. Enable **Internet access**: Settings → Internet → On  
2. Enable **GPU**: Settings → Accelerator → GPU T4  
3. Run cells **in order** from top to bottom ↓

## 1. Install Dependencies

In [1]:
import subprocess, sys

# Install required packages that are not pre-installed in the Kaggle environment
packages = ["torchgeo", "pytorch-lightning", "torchmetrics", "grad-cam"]
for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

print("All dependencies installed successfully.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.1/688.1 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.6/863.6 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.5/786.5 kB 41.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 58.6 MB/s eta 0:00:00
All dependencies installed successfully.


## 2. Imports

In [2]:
from __future__ import annotations
import os, sys, random, glob
from typing import Any, Dict, Optional

import matplotlib.pyplot as plt
import numpy as np
import pytorch_lightning as pl
import torch
import torch.nn as nn
from matplotlib.patches import Patch
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from torchgeo.datasets import EuroSAT
from torchmetrics import AUROC, F1Score, Precision, Recall
from torchmetrics.classification import BinaryPrecisionRecallCurve
from torchvision import models, transforms

print(f"PyTorch    version : {torch.__version__}")
print(f"Lightning  version : {pl.__version__}")
print(f"CUDA available     : {torch.cuda.is_available()}")

PyTorch    version : 2.10.0+cu128
Lightning  version : 2.6.1
CUDA available     : True


## 3. Configuration & Dataset Download

EuroSAT is downloaded automatically by TorchGeo (~2 GB, first run only).  
All three official splits (`train` / `val` / `test`) are fetched separately  
because each split file is downloaded on demand.

In [3]:
# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
pl.seed_everything(SEED)

# ── EuroSAT class index that maps to the positive label (water = 1) ───────────
# Index 8 = River. Add index 9 (SeaLake) to broaden the positive class.
WATER_CLASS_INDICES = {8}

# ── Training hyper-parameters ─────────────────────────────────────────────────
BATCH_SIZE    = 32
NUM_WORKERS   = 2
MAX_EPOCHS    = 15
LEARNING_RATE = 1e-4
WEIGHT_DECAY  = 1e-4

# ── Imbalance handling strategy ───────────────────────────────────────────────
# False → use pos_weight in BCEWithLogitsLoss  (default, recommended)
# True  → use WeightedRandomSampler on the training loader
USE_SAMPLER = False

# ── Visualisation parameters ──────────────────────────────────────────────────
VIS_N_GRADCAM = 6    # patches shown in the GradCAM figure (half water, half other)
VIS_N_GRID    = 64   # patches shown in the confidence grid

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_ROOT = "/kaggle/working/eurosat_data"
CKPT_DIR  = "/kaggle/working/checkpoints"
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(CKPT_DIR,  exist_ok=True)

# ── Download all three splits ─────────────────────────────────────────────────
# TorchGeo downloads each split file (eurosat-train.txt, etc.) on first request.
# The image archive (EuroSATallBands.zip) is shared across all splits.
for split in ["train", "val", "test"]:
    split_file = os.path.join(DATA_ROOT, EuroSAT.split_filenames[split])
    if not os.path.isfile(split_file):
        print(f"Downloading '{split}' split ...")
        _tmp = EuroSAT(root=DATA_ROOT, split=split, download=True)
        del _tmp
    else:
        print(f"Split '{split}' already present — skipping download.")

# ── Verify the expected files exist ──────────────────────────────────────────
print("\nVerification:")
all_ok = True
for split, fname in EuroSAT.split_filenames.items():
    exists = os.path.isfile(os.path.join(DATA_ROOT, fname))
    print(f"  {fname:30s} exists = {exists}")
    all_ok = all_ok and exists

base_exists = os.path.isdir(os.path.join(DATA_ROOT, EuroSAT.base_dir))
print(f"  image directory               exists = {base_exists}")

if all_ok and base_exists:
    print(f"\nDataset ready at: {DATA_ROOT}")
else:
    raise RuntimeError("Dataset verification failed. Check the paths above.")

Seed set to 42



Verification:
  eurosat-train.txt              exists = True
  eurosat-val.txt                exists = True
  eurosat-test.txt               exists = True
  image directory               exists = True

Dataset ready at: /kaggle/working/eurosat_data


## 4. Dataset Wrapper — `BinaryEuroSAT`

Subclasses `torchgeo.datasets.EuroSAT` and applies three transformations on every `__getitem__` call:

1. **Band selection** — loads only the 3 RGB bands (B04, B03, B02) via `bands=EuroSAT.rgb_bands`
2. **Label remapping** — River (class index 8) → `1.0`; all other classes → `0.0`  
3. **Pixel normalisation** — divides uint16 Sentinel-2 values by 10 000 to bring them into [0, 1], then applies per-band mean/std normalisation

> **Note on class-level transform:** `_base_transform` is defined at the class level (not inside `__init__`) to avoid a `RecursionError` caused by TorchGeo calling `__getitem__` internally during initialisation before instance attributes are set.

In [4]:
class BinaryEuroSAT(EuroSAT):
    """
    EuroSAT wrapper for binary water / non-water classification.

    Modifications over the base class
    ----------------------------------
    * Loads RGB bands only (B04, B03, B02).
    * Remaps the 10-class label to binary:  River (8) → 1, everything else → 0.
    * Normalises pixel values using EuroSAT-specific per-band statistics.

    Adaptation for 13-band multispectral input
    -------------------------------------------
    1. Remove ``bands=EuroSAT.rgb_bands`` from ``super().__init__()`` to load all 13 bands.
    2. Replace ``_MEAN`` / ``_STD`` with 13-band EuroSAT-MS statistics.
    3. In ``build_resnet50()``, pass ``in_channels=13``; the function handles
       replacing ``conv1`` with a warm-started 13-channel convolution.
    """

    # Per-band statistics for EuroSAT RGB (Sentinel-2, reflectance scaled to [0, 1])
    _MEAN = [0.3444, 0.3803, 0.4078]   # B04 (Red), B03 (Green), B02 (Blue)
    _STD  = [0.2035, 0.1354, 0.1155]

    # Defined at class level to avoid RecursionError during TorchGeo's __init__
    _base_transform = transforms.Normalize(
        mean=[0.3444, 0.3803, 0.4078],
        std= [0.2035, 0.1354, 0.1155],
    )

    def __init__(self, root: str, split: str = "train", download: bool = False):
        super().__init__(
            root=root,
            split=split,
            download=download,
            bands=EuroSAT.rgb_bands,   # RGB only — remove for 13-band MS
        )

    def __getitem__(self, index: int) -> Dict[str, Any]:
        sample = super().__getitem__(index)

        # Remap 10-class label to binary float (required by BCEWithLogitsLoss)
        sample["label"] = torch.tensor(
            1 if int(sample["label"]) in WATER_CLASS_INDICES else 0,
            dtype=torch.float32,
        )

        # Scale uint16 reflectance to [0, 1] then normalise
        sample["image"] = self._base_transform(
            sample["image"].float() / 10000.0
        )
        return sample


# ── Sanity check ──────────────────────────────────────────────────────────────
_ds = BinaryEuroSAT(DATA_ROOT, split="train", download=False)
_s  = _ds[0]
print(f"Dataset loaded successfully.")
print(f"  Image tensor shape : {_s['image'].shape}   (C x H x W)")
print(f"  Label              : {int(_s['label'].item())}  (0 = other, 1 = water)")
print(f"  Training set size  : {len(_ds)} images")
del _ds

Dataset loaded successfully.
  Image tensor shape : torch.Size([3, 64, 64])   (C x H x W)
  Label              : 0  (0 = other, 1 = water)
  Training set size  : 16200 images


## 5. Data Module — `EuroSATDataModule`

Uses the three pre-defined TorchGeo splits (`train` / `val` / `test`) directly.

**Class imbalance:** Water patches represent roughly 10 % of the training set.  
`pos_weight = n_negatives / n_positives` is computed at setup time and passed to `BCEWithLogitsLoss`, making the model penalise false negatives proportionally more.  
Alternatively, set `USE_SAMPLER = True` to use `WeightedRandomSampler` instead.

In [5]:
class EuroSATDataModule(pl.LightningDataModule):
    """
    Lightning DataModule for EuroSAT binary water classification.

    Uses the official TorchGeo train / val / test splits.
    Computes pos_weight from the training set to address the ~10 % water
    class imbalance, and optionally uses WeightedRandomSampler instead.
    """

    def __init__(
        self,
        root: str        = DATA_ROOT,
        batch_size: int  = BATCH_SIZE,
        num_workers: int = NUM_WORKERS,
        use_sampler: bool = USE_SAMPLER,
    ):
        super().__init__()
        self.root        = root
        self.batch_size  = batch_size
        self.num_workers = num_workers
        self.use_sampler = use_sampler
        self.train_dataset = self.val_dataset = self.test_dataset = None
        self.pos_weight    = None

    def setup(self, stage: Optional[str] = None):
        self.train_dataset = BinaryEuroSAT(self.root, split="train", download=False)
        self.val_dataset   = BinaryEuroSAT(self.root, split="val",   download=False)
        self.test_dataset  = BinaryEuroSAT(self.root, split="test",  download=False)

        # Compute class frequencies on the training set for pos_weight
        labels = torch.tensor([
            int(self.train_dataset[i]["label"].item())
            for i in range(len(self.train_dataset))
        ])
        n_pos = labels.sum().item()
        n_neg = len(labels) - n_pos
        self.pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32)

        print(f"Split sizes  →  train: {len(self.train_dataset)} | "
              f"val: {len(self.val_dataset)} | test: {len(self.test_dataset)}")
        print(f"Water samples: {int(n_pos)} ({100 * n_pos / len(labels):.1f}% of train)")
        print(f"pos_weight   : {self.pos_weight.item():.2f}  "
              f"(loss penalty applied to the minority class)")

    def _build_sampler(self) -> Optional[WeightedRandomSampler]:
        """Over-samples water patches so each mini-batch is approximately balanced."""
        if not self.use_sampler:
            return None
        labels = torch.tensor([
            int(self.train_dataset[i]["label"].item())
            for i in range(len(self.train_dataset))
        ])
        class_weights  = 1.0 / torch.bincount(labels.long()).float()
        sample_weights = class_weights[labels.long()]
        return WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

    def train_dataloader(self) -> DataLoader:
        sampler = self._build_sampler()
        return DataLoader(
            self.train_dataset, batch_size=self.batch_size,
            sampler=sampler, shuffle=(sampler is None),
            num_workers=self.num_workers, pin_memory=True,
        )

    def val_dataloader(self) -> DataLoader:
        return DataLoader(
            self.val_dataset, batch_size=self.batch_size,
            shuffle=False, num_workers=self.num_workers, pin_memory=True,
        )

    def test_dataloader(self) -> DataLoader:
        return DataLoader(
            self.test_dataset, batch_size=self.batch_size,
            shuffle=False, num_workers=self.num_workers, pin_memory=True,
        )

print("EuroSATDataModule defined.")

EuroSATDataModule defined.


## 6. Model — `WaterClassifier`

**Backbone:** ResNet-50 pre-trained on ImageNet (`IMAGENET1K_V2` weights).  
The final fully-connected layer is replaced with a single linear unit that outputs one raw logit, compatible with `BCEWithLogitsLoss`.

**Loss function:** `BCEWithLogitsLoss` with `pos_weight` to address class imbalance.  
When `USE_SAMPLER = True`, the sampler already balances the batches and `pos_weight` is set to 1.

**Metrics reported:**
- **Validation:** F1-Score, Precision, Recall (logged each epoch; F1 is the checkpoint monitor)
- **Test:** F1-Score, Precision, Recall, AUROC, full Precision-Recall curve

**Optimiser:** AdamW with cosine annealing learning-rate schedule.

In [6]:
def build_resnet50(in_channels: int = 3) -> nn.Module:
    """
    Constructs a ResNet-50 with a single-logit output head for binary classification.

    Parameters
    ----------
    in_channels : int
        Number of input channels. Use 3 for RGB EuroSAT (default).
        Use 13 for the full multispectral EuroSAT variant: the first
        convolutional layer is replaced with a new Conv2d(13, 64, ...)
        whose extra channels are warm-started from the mean of the RGB weights.
    """
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

    if in_channels != 3:
        old_conv = model.conv1
        new_conv = nn.Conv2d(in_channels, 64, kernel_size=7,
                             stride=2, padding=3, bias=False)
        with torch.no_grad():
            new_conv.weight[:, :3] = old_conv.weight
            # Warm-start extra channels from the mean of the RGB weights
            new_conv.weight[:, 3:] = old_conv.weight.mean(
                dim=1, keepdim=True).expand(-1, in_channels - 3, -1, -1)
        model.conv1 = new_conv

    # Replace the 1000-class head with a single logit for binary classification
    model.fc = nn.Linear(model.fc.in_features, 1)
    return model


class WaterClassifier(pl.LightningModule):
    """
    PyTorch Lightning module for binary water / non-water classification.

    Architecture  : ResNet-50 → single linear logit
    Loss          : BCEWithLogitsLoss with pos_weight for class imbalance
    Optimiser     : AdamW + CosineAnnealingLR
    """

    def __init__(
        self,
        pos_weight: torch.Tensor,
        in_channels: int    = 3,
        lr: float           = LEARNING_RATE,
        weight_decay: float = WEIGHT_DECAY,
        use_sampler: bool   = USE_SAMPLER,
    ):
        super().__init__()
        self.save_hyperparameters(ignore=["pos_weight"])

        self.model = build_resnet50(in_channels)

        # When using WeightedRandomSampler the batches are already balanced,
        # so we set pos_weight = 1 to avoid double-counting the correction.
        effective_pw   = torch.ones(1) if use_sampler else pos_weight
        self.criterion = nn.BCEWithLogitsLoss(pos_weight=effective_pw)

        # ── Metrics ───────────────────────────────────────────────────────────
        kw = dict(task="binary", threshold=0.5)
        self.val_f1    = F1Score(**kw);  self.val_prec = Precision(**kw)
        self.val_rec   = Recall(**kw)
        self.test_f1   = F1Score(**kw);  self.test_prec = Precision(**kw)
        self.test_rec  = Recall(**kw);   self.test_auroc = AUROC(task="binary")
        self.test_pr   = BinaryPrecisionRecallCurve()
        self.pr_curve_data = None   # populated in on_test_epoch_end for inspection

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x).squeeze(1)   # (B, 1) → (B,)

    def _shared_step(self, batch: Dict[str, Any]):
        logits = self(batch["image"])
        labels = batch["label"]
        loss   = self.criterion(logits, labels)
        probs  = torch.sigmoid(logits)
        return loss, probs, labels

    # ── Training ──────────────────────────────────────────────────────────────
    def training_step(self, batch: Dict[str, Any], _: int) -> torch.Tensor:
        loss, _, _ = self._shared_step(batch)
        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    # ── Validation ────────────────────────────────────────────────────────────
    def validation_step(self, batch: Dict[str, Any], _: int):
        loss, probs, labels = self._shared_step(batch)
        li = labels.int()
        self.val_f1.update(probs, li)
        self.val_prec.update(probs, li)
        self.val_rec.update(probs, li)
        self.log("val_loss", loss, prog_bar=True)

    def on_validation_epoch_end(self):
        self.log("val_f1",        self.val_f1.compute(),   prog_bar=True)
        self.log("val_precision", self.val_prec.compute())
        self.log("val_recall",    self.val_rec.compute())
        self.val_f1.reset(); self.val_prec.reset(); self.val_rec.reset()

    # ── Test ──────────────────────────────────────────────────────────────────
    def test_step(self, batch: Dict[str, Any], _: int):
        _, probs, labels = self._shared_step(batch)
        li = labels.int()
        self.test_f1.update(probs, li);    self.test_prec.update(probs, li)
        self.test_rec.update(probs, li);   self.test_auroc.update(probs, li)
        self.test_pr.update(probs, li)

    def on_test_epoch_end(self):
        f1    = self.test_f1.compute()
        prec  = self.test_prec.compute()
        rec   = self.test_rec.compute()
        auroc = self.test_auroc.compute()
        pv, rv, tv = self.test_pr.compute()

        # Write directly to the real stdout to bypass the Rich/Jupyter
        # recursion issue that occurs when printing inside Lightning hooks.
        out = sys.__stdout__
        out.write(f"\n{'='*45}\nTEST SET RESULTS\n{'='*45}\n")
        out.write(f"  F1-Score  : {f1:.4f}\n")
        out.write(f"  Precision : {prec:.4f}\n")
        out.write(f"  Recall    : {rec:.4f}\n")
        out.write(f"  AUROC     : {auroc:.4f}\n{'='*45}\n\n")
        out.flush()

        # Store the full PR curve for post-training inspection
        self.pr_curve_data = {"precision": pv.cpu(),
                              "recall":    rv.cpu(),
                              "thresholds": tv.cpu()}

        self.log_dict({"test_f1": f1, "test_precision": prec,
                       "test_recall": rec, "test_auroc": auroc})

        self.test_f1.reset();  self.test_prec.reset()
        self.test_rec.reset(); self.test_auroc.reset(); self.test_pr.reset()

    # ── Optimiser ─────────────────────────────────────────────────────────────
    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=self.hparams.lr,
            weight_decay=self.hparams.weight_decay,
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=MAX_EPOCHS, eta_min=1e-6,
        )
        return {"optimizer": optimizer,
                "lr_scheduler": {"scheduler": scheduler, "monitor": "val_loss"}}


print("WaterClassifier defined.")

WaterClassifier defined.


## 7. Training

The trainer uses three callbacks:
- **`ModelCheckpoint`** — saves the weights whenever `val_f1` improves
- **`EarlyStopping`** — halts training if `val_f1` does not improve for 5 consecutive epochs
- **`LearningRateMonitor`** — logs the current LR to verify cosine decay is working

Expected runtime on a single GPU T4: **30–60 minutes** (15 epochs, early stopping may trigger sooner).

In [ ]:
from pytorch_lightning.callbacks import (
    EarlyStopping, LearningRateMonitor, ModelCheckpoint,
)

# ── Initialise data and model ─────────────────────────────────────────────────
dm = EuroSATDataModule()
dm.setup()

model = WaterClassifier(pos_weight=dm.pos_weight)

# ── Configure trainer ─────────────────────────────────────────────────────────
trainer = pl.Trainer(
    max_epochs        = MAX_EPOCHS,
    accelerator       = "auto",       # GPU if available, else CPU
    devices           = 1,
    callbacks         = [
        ModelCheckpoint(
            dirpath    = CKPT_DIR,
            monitor    = "val_f1",
            mode       = "max",
            save_top_k = 1,
            filename   = "best-{epoch:02d}-{val_f1:.4f}",
            verbose    = True,
        ),
        EarlyStopping(
            monitor  = "val_f1",
            mode     = "max",
            patience = 5,
            verbose  = True,
        ),
        LearningRateMonitor(logging_interval="epoch"),
    ],
    log_every_n_steps = 20,
    deterministic     = True,         # ensures reproducibility
)

# ── Train ─────────────────────────────────────────────────────────────────────
print("Starting training ...")
trainer.fit(model, datamodule=dm)

# ── Evaluate on the held-out test set using the best checkpoint ───────────────
print("\nEvaluating on test set ...")
trainer.test(model, datamodule=dm, ckpt_path="best")

BEST_CKPT = trainer.checkpoint_callback.best_model_path
print(f"\nBest checkpoint saved to: {BEST_CKPT}")

Split sizes  →  train: 16200 | val: 5400 | test: 5400
Water samples: 1460 (9.0% of train)
pos_weight   : 10.10  (loss penalty applied to the minority class)
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 173MB/s] 
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Starting training ...


2026-03-27 16:51:28.745745: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774630288.942251      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774630289.000233      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774630289.496297      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774630289.496321      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774630289.496338      55 computation_placer.cc:177] computation placer alr

## 8. Load Best Checkpoint

Loads the checkpoint with the highest validation F1-score.  
If the kernel was restarted after training, the checkpoint is located automatically.

In [ ]:
# If the kernel was restarted, locate the checkpoint automatically
if "BEST_CKPT" not in dir() or not BEST_CKPT:
    available = sorted(glob.glob(f"{CKPT_DIR}/*.ckpt"))
    if not available:
        raise FileNotFoundError(
            f"No checkpoint found in {CKPT_DIR}.\n"
            "Please run Cell 7 (training) first."
        )
    BEST_CKPT = available[-1]

print(f"Loading checkpoint: {BEST_CKPT}")

best_model = WaterClassifier.load_from_checkpoint(
    BEST_CKPT,
    pos_weight=torch.ones(1),   # pos_weight is not needed for inference
)
best_model.eval()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
best_model = best_model.to(DEVICE)
print(f"Model loaded on {DEVICE}.")

## 9. Visualisation — GradCAM

**Gradient-weighted Class Activation Mapping (GradCAM)** back-propagates gradients through the last convolutional block of the ResNet-50 (`layer4`) to produce a spatial heatmap showing which pixels most influenced the prediction.

Three columns are shown per sample:
1. Original de-normalised RGB patch
2. GradCAM heatmap (jet colourmap — red = high activation)
3. Heatmap blended onto the original image

The border colour of each row indicates ground truth: **green = water**, **red = other**.

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import BinaryClassifierOutputTarget


def denormalise(tensor: torch.Tensor) -> np.ndarray:
    """Invert EuroSAT normalisation: (3, H, W) tensor → (H, W, 3) float32 in [0, 1]."""
    mean = torch.tensor([0.3444, 0.3803, 0.4078]).view(3, 1, 1)
    std  = torch.tensor([0.2035, 0.1354, 0.1155]).view(3, 1, 1)
    img  = (tensor.cpu() * std + mean).permute(1, 2, 0).numpy()
    return np.clip(img, 0, 1).astype(np.float32)


# ── Collect 3 water and 3 non-water patches from the test split ───────────────
ds_test = BinaryEuroSAT(DATA_ROOT, split="test", download=False)
water_samples, other_samples = [], []

for idx in random.sample(range(len(ds_test)), len(ds_test)):
    sample = ds_test[idx]
    if sample["label"].item() == 1 and len(water_samples) < 3:
        water_samples.append(sample)
    if sample["label"].item() == 0 and len(other_samples) < 3:
        other_samples.append(sample)
    if len(water_samples) == 3 and len(other_samples) == 3:
        break

print(f"Collected {len(water_samples)} water + {len(other_samples)} non-water patches.")

# ── GradCAM setup — target the last residual block of ResNet-50 ───────────────
cam = GradCAM(model=best_model.model,
              target_layers=[best_model.model.layer4[-1]])

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(6, 3, figsize=(10, 20), facecolor="#0d1117")
fig.suptitle("GradCAM — Spatial Focus of the Water Classifier",
             color="white", fontsize=15, fontweight="bold", y=1.005)

for col, title in enumerate(["Input patch (RGB)", "GradCAM heatmap", "Overlay"]):
    axes[0, col].set_title(title, color="#58a6ff", fontsize=11, pad=10)

for row, sample in enumerate(water_samples + other_samples):
    img_tensor = sample["image"].unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        prob = torch.sigmoid(best_model(img_tensor)).item()

    heatmap = cam(input_tensor=img_tensor,
                  targets=[BinaryClassifierOutputTarget(1)])[0]   # (H, W)
    rgb     = denormalise(sample["image"])
    overlay = show_cam_on_image(rgb, heatmap, use_rgb=True)

    gt_label  = "WATER" if sample["label"].item() == 1 else "OTHER"
    gt_colour = "#39d353" if sample["label"].item() == 1 else "#f85149"
    pred      = "WATER"  if prob >= 0.5 else "OTHER"

    axes[row, 0].imshow(rgb)
    axes[row, 1].imshow(heatmap, cmap="jet", vmin=0, vmax=1)
    axes[row, 2].imshow(overlay)

    for col in range(3):
        axes[row, col].axis("off")
        for spine in axes[row, col].spines.values():
            spine.set_edgecolor(gt_colour)
            spine.set_linewidth(2.5)

    axes[row, 0].text(2, 6, f"GT: {gt_label}", color=gt_colour,
                      fontsize=9, fontweight="bold",
                      bbox=dict(facecolor="#0d1117", alpha=0.8,
                                pad=2, edgecolor="none"))
    axes[row, 0].text(2, 14, f"P = {prob:.2f}  →  {pred}", color="#e3b341",
                      fontsize=9,
                      bbox=dict(facecolor="#0d1117", alpha=0.8,
                                pad=2, edgecolor="none"))

plt.tight_layout(pad=1.5)
plt.savefig("/kaggle/working/fig_gradcam.png", dpi=150,
            bbox_inches="tight", facecolor="#0d1117")
plt.show()
print("Saved: /kaggle/working/fig_gradcam.png")

## 10. Visualisation — Raw vs. Preprocessed Input

Illustrates the preprocessing pipeline applied before each image is fed to the model:

- **Left column:** de-normalised RGB approximation — the closest representation to what the raw patch looks like visually
- **Right column:** false-colour rendering of the normalised red channel (RdYlBu_r colourmap) — shows the numerical values the model actually operates on

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(7, 13), facecolor="#0d1117")
fig.suptitle("Raw Input  vs.  Normalised Input (as seen by the model)",
             color="white", fontsize=13, fontweight="bold", y=1.005)
axes[0, 0].set_title("RGB (de-normalised)",           color="#58a6ff", fontsize=10, pad=8)
axes[0, 1].set_title("False-colour — red band (B04)", color="#58a6ff", fontsize=10, pad=8)

for row, sample in enumerate(water_samples[:2] + other_samples[:2]):
    rgb       = denormalise(sample["image"])
    gt_label  = "WATER" if sample["label"].item() == 1 else "OTHER"
    gt_colour = "#39d353" if sample["label"].item() == 1 else "#f85149"

    axes[row, 0].imshow(rgb)
    axes[row, 1].imshow(rgb[:, :, 0], cmap="RdYlBu_r", vmin=0, vmax=1)

    for col in range(2):
        axes[row, col].axis("off")

    axes[row, 0].text(2, 6, gt_label, color=gt_colour,
                      fontsize=9, fontweight="bold",
                      bbox=dict(facecolor="#0d1117", alpha=0.8,
                                pad=2, edgecolor="none"))

plt.tight_layout(pad=1.5)
plt.savefig("/kaggle/working/fig_raw_vs_preprocessed.png", dpi=150,
            bbox_inches="tight", facecolor="#0d1117")
plt.show()
print("Saved: /kaggle/working/fig_raw_vs_preprocessed.png")

## 11. Visualisation — Prediction Confidence Grid

Runs inference on 64 randomly sampled test patches and arranges them in a square grid.

- **Border colour** encodes `P(water)` continuously: bright green → high confidence water; bright red → high confidence non-water
- **Corner marker** shows ground truth: `●` = water, `○` = other
- Misclassified patches are easy to spot: a red-bordered patch with a `●` marker indicates a missed water body (false negative)

In [ ]:
n_patches = min(VIS_N_GRID, len(ds_test))
samples   = [ds_test[i] for i in random.sample(range(len(ds_test)), n_patches)]

image_batch = torch.stack([s["image"] for s in samples]).to(DEVICE)
gt_labels   = torch.tensor([int(s["label"].item()) for s in samples])

with torch.no_grad():
    pred_probs = torch.sigmoid(best_model(image_batch)).cpu().numpy()

n_cols = int(np.ceil(np.sqrt(n_patches)))
n_rows = int(np.ceil(n_patches / n_cols))

fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(n_cols * 1.5, n_rows * 1.5 + 1.0),
                         facecolor="#0d1117")
fig.suptitle("Prediction Confidence Grid  ·  Border colour = P(water)",
             color="white", fontsize=12, fontweight="bold")

for i in range(n_rows * n_cols):
    ax = axes[i // n_cols, i % n_cols]
    ax.axis("off")
    if i >= n_patches:
        continue

    rgb   = denormalise(samples[i]["image"])
    prob  = float(pred_probs[i])
    gt    = int(gt_labels[i].item())

    ax.imshow(rgb)

    # Colour border by predicted probability: green = high P(water), red = low
    border_colour = (0.85 * (1 - prob), 0.85 * prob, 0.1)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor(border_colour)
        spine.set_linewidth(3)

    # Ground-truth marker in the top-right corner
    ax.text(0.97, 0.97, "●" if gt == 1 else "○",
            transform=ax.transAxes,
            color="#39d353" if gt == 1 else "#888",
            fontsize=7, ha="right", va="top")

fig.legend(
    handles=[
        Patch(facecolor=(0, 0.85, 0.1),   label="High P(water)"),
        Patch(facecolor=(0.85, 0, 0.1),   label="Low P(water)"),
        Patch(facecolor="#0d1117", edgecolor="#39d353",
              linewidth=2, label="GT = Water (●)"),
        Patch(facecolor="#0d1117", edgecolor="#888",
              linewidth=2, label="GT = Other (○)"),
    ],
    loc="lower center", ncol=4, fontsize=9,
    facecolor="#161b22", edgecolor="#30363d", labelcolor="white",
    bbox_to_anchor=(0.5, -0.02),
)

plt.tight_layout(pad=0.4)
plt.savefig("/kaggle/working/fig_confidence_grid.png", dpi=150,
            bbox_inches="tight", facecolor="#0d1117")
plt.show()
print("Saved: /kaggle/working/fig_confidence_grid.png")

## 12. Real-World Inference — Sliding Window

Demonstrates how the patch classifier can be applied to a **full satellite scene** that is larger than 64×64 pixels.

**Pipeline:**
1. A synthetic scene (512×512) is generated with a diagonal river, an elliptical lake, urban area, and vegetation background
2. A 64×64 sliding window with stride 32 extracts overlapping patches
3. The classifier assigns `P(water)` to each patch
4. Overlapping predictions are **averaged**, producing a smooth full-resolution probability map with no tile-boundary artefacts

**Output panels:**
- Original RGB scene
- P(water) heatmap (Blues colourmap)
- Binary mask at threshold 0.5
- Heatmap blended onto the original scene

> To use a real Sentinel-2 scene, replace `scene` with your own `numpy.ndarray` of shape `(H, W, 3)`, `float32`, values in `[0, 1]`.

In [ ]:
# ── Generate a synthetic satellite scene for demonstration ───────────────────
rng = np.random.default_rng(42)
H, W = 512, 512
scene = np.zeros((H, W, 3), dtype=np.float32)

# Background: green vegetation
scene[:, :, 0] = rng.uniform(0.05, 0.15, (H, W))
scene[:, :, 1] = rng.uniform(0.20, 0.40, (H, W))
scene[:, :, 2] = rng.uniform(0.05, 0.15, (H, W))

# Urban area: upper-right quadrant (higher, more uniform reflectance)
scene[:H//2, W//2:] = rng.uniform(0.30, 0.55, (H//2, W//2, 3))

# River: diagonal blue band crossing the scene
for y in range(H):
    x_centre = W // 2 + int((y - H // 2) * 0.35)
    x0, x1   = max(0, x_centre - 22), min(W, x_centre + 22)
    scene[y, x0:x1, 0] = rng.uniform(0.03, 0.10, x1 - x0)
    scene[y, x0:x1, 1] = rng.uniform(0.08, 0.18, x1 - x0)
    scene[y, x0:x1, 2] = rng.uniform(0.28, 0.45, x1 - x0)

# Lake: ellipse in the lower-left area
yy, xx = np.ogrid[:H, :W]
lake_mask = ((yy - 390) / 60) ** 2 + ((xx - 110) / 85) ** 2 <= 1
scene[lake_mask, 0] = rng.uniform(0.03, 0.09, lake_mask.sum())
scene[lake_mask, 1] = rng.uniform(0.07, 0.16, lake_mask.sum())
scene[lake_mask, 2] = rng.uniform(0.30, 0.48, lake_mask.sum())

# Add low-amplitude noise for texture
scene = np.clip(scene + rng.normal(0, 0.01, scene.shape), 0, 1).astype(np.float32)

# ── Sliding window inference ──────────────────────────────────────────────────
PATCH_SIZE = 64
STRIDE     = 32

normalise = transforms.Normalize(
    mean=[0.3444, 0.3803, 0.4078],
    std= [0.2035, 0.1354, 0.1155],
)

patches, positions = [], []
for r in range(0, H - PATCH_SIZE + 1, STRIDE):
    for c in range(0, W - PATCH_SIZE + 1, STRIDE):
        patch = torch.from_numpy(
            scene[r:r + PATCH_SIZE, c:c + PATCH_SIZE]
        ).permute(2, 0, 1)
        patches.append(normalise(patch))
        positions.append((r, c))

print(f"Total patches extracted: {len(patches)}")

# Run inference in mini-batches of 128
INFER_BATCH = 128
all_probs   = []
for i in range(0, len(patches), INFER_BATCH):
    batch = torch.stack(patches[i:i + INFER_BATCH]).to(DEVICE)
    with torch.no_grad():
        all_probs.append(torch.sigmoid(best_model(batch)).cpu().numpy())

patch_probs = np.concatenate(all_probs)   # (N,)

# ── Assemble probability map by averaging overlapping predictions ─────────────
accumulator = np.zeros((H, W), dtype=np.float32)
count_map   = np.zeros((H, W), dtype=np.float32)

for prob, (r, c) in zip(patch_probs, positions):
    accumulator[r:r + PATCH_SIZE, c:c + PATCH_SIZE] += prob
    count_map  [r:r + PATCH_SIZE, c:c + PATCH_SIZE] += 1

prob_map   = accumulator / np.where(count_map == 0, 1, count_map)
water_pct  = 100 * (prob_map >= 0.5).mean()
print(f"Mean P(water): {prob_map.mean():.3f} | "
      f"Pixels classified as water: {water_pct:.1f}%")

# ── Four-panel summary figure ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(22, 5.5), facecolor="#0d1117")
fig.suptitle("Real-World Water Detection via Sliding-Window Inference  (64×64 patches)",
             color="white", fontsize=14, fontweight="bold")

# Panel 1: original scene
axes[0].imshow(scene)
axes[0].set_title("Synthetic RGB scene", color="#58a6ff", fontsize=11)

# Panel 2: probability heatmap
im = axes[1].imshow(prob_map, cmap="Blues", vmin=0, vmax=1)
axes[1].set_title("P(water) heatmap", color="#58a6ff", fontsize=11)
cb = fig.colorbar(im, ax=axes[1], fraction=0.04, pad=0.03)
cb.set_label("P(water)", color="white")
cb.ax.yaxis.set_tick_params(color="white")
plt.setp(cb.ax.yaxis.get_ticklabels(), color="white")

# Panel 3: binary mask at threshold 0.5
axes[2].imshow(prob_map >= 0.5, cmap="RdYlGn", vmin=0, vmax=1)
axes[2].set_title(f"Binary mask  (t = 0.5)  |  {water_pct:.1f}% water",
                  color="#58a6ff", fontsize=11)
axes[2].legend(
    handles=[Patch(facecolor="#00c853", label="Water"),
             Patch(facecolor="#b71c1c", label="Other")],
    loc="lower right", fontsize=9,
    facecolor="#161b22", edgecolor="#30363d", labelcolor="white",
)

# Panel 4: heatmap blended onto the original scene
blue_layer = np.stack([np.zeros((H, W)), prob_map * 0.4, prob_map], axis=-1)
overlay    = np.clip(
    (1 - 0.6 * prob_map[:, :, None]) * scene
    + 0.6 * prob_map[:, :, None] * blue_layer,
    0, 1,
)
axes[3].imshow(overlay)
axes[3].set_title("Overlay  (blue tint = predicted water)", color="#58a6ff", fontsize=11)

for ax in axes:
    ax.axis("off")

plt.tight_layout(pad=1.0)
plt.savefig("/kaggle/working/fig_realworld.png", dpi=150,
            bbox_inches="tight", facecolor="#0d1117")
plt.show()
print("Saved: /kaggle/working/fig_realworld.png")

## 13. Summary of Output Files

In [ ]:
output_files = sorted(
    glob.glob("/kaggle/working/fig_*.png") +
    glob.glob(f"{CKPT_DIR}/*.ckpt")
)

print("Output files written to /kaggle/working/")
print("-" * 60)
for path in output_files:
    size_kb = os.path.getsize(path) / 1024
    print(f"  {os.path.basename(path):<45}  {size_kb:>8.0f} KB")

print("\nDownload from the Output panel on the right →")